In [1]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

DATA = Path.cwd().parent / "data" / "cardio_clean.csv"
df = pd.read_csv(DATA)

FEATURES = ["age_years", "gender", "ap_hi", "ap_lo", "cholesterol",
            "gluc", "smoke", "alco", "active", "bmi"]
X = df[FEATURES]
y = df["cardio"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Training patients:", len(X_train))
print("Testing patients: ", len(X_test))

Training patients: 54864
Testing patients:  13717


In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train, y_train)

pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, pred):.1%}\n")
print(classification_report(y_test, pred, target_names=["No disease", "Disease"]))

Accuracy: 73.3%

              precision    recall  f1-score   support

  No disease       0.71      0.79      0.75      6931
     Disease       0.76      0.67      0.71      6786

    accuracy                           0.73     13717
   macro avg       0.74      0.73      0.73     13717
weighted avg       0.74      0.73      0.73     13717



In [3]:
importance = pd.Series(model.feature_importances_, index=FEATURES)
print((importance.sort_values(ascending=False) * 100).round(1).to_string())

ap_hi          50.4
ap_lo          18.2
age_years      13.8
cholesterol     9.5
bmi             5.4
gluc            1.1
active          0.7
gender          0.4
smoke           0.3
alco            0.2


In [4]:
import joblib

MODEL_PATH = Path.cwd().parent / "models" / "cardio_rf.joblib"
joblib.dump({"model": model, "features": FEATURES}, MODEL_PATH)
print("Saved:", MODEL_PATH.name, f"({MODEL_PATH.stat().st_size / 1e6:.1f} MB)")

Saved: cardio_rf.joblib (15.2 MB)
